# 02 — Results analysis

Explore `results/validation_metrics.json`, the downscaled NetCDF and the trained models after `python main.py validate`.

In [ ]:
import sys, json, numpy as np, pandas as pd, xarray as xr, matplotlib.pyplot as plt
from IPython.display import Image, display
sys.path.insert(0, '..')
from src.utils import load_config, grids_from_config
from src.validation import SEQ_CMAP, DIV_CMAP, METHODS, LABELS
cfg = load_config('../config.yaml'); grids = grids_from_config(cfg)
res = json.load(open(f"{cfg['paths']['results']}/validation_metrics.json"))
print('Period', res['period'], 'days', res['n_days'], 'method', res['method'])
pd.DataFrame(res['comparison_table']).T.rename(columns=LABELS).round(3)

## Success criteria (plan §7)

In [ ]:
pd.DataFrame(res['success_criteria']).T

## Skill by intensity class and by season

In [ ]:
rows = {(m, c): d for m in METHODS for c, d in res['methods'][m]['by_intensity'].items()}
df = pd.DataFrame(rows).T[['n', 'bias', 'rmse', 'pearson_r', 'nse']]
df.index.names = ['method', 'class']; df.round(3)

In [ ]:
seas = pd.DataFrame({m: {s: d['rmse'] for s, d in res['methods'][m]['by_season'].items()} for m in METHODS}).rename(columns=LABELS)
seas.plot.bar(figsize=(8, 3.5), color=['#2a78d6', '#eb6834', '#1baf7a'], width=0.75); plt.ylabel('RMSE (mm/day)'); plt.xticks(rotation=0);

## Figures produced by the validation step

In [ ]:
rdir = cfg['paths']['results']
for f in ['spatial_plots/bias_rmse_maps.png', 'spatial_plots/nse_maps.png', 'qq_plot.png', 'timeseries_comparisons/timeseries_locations.png', 'feature_importance_stage1_10km.png', 'feature_importance_stage2_1km.png']:
    try:
        display(Image(filename=f'{rdir}/{f}', width=1000))
    except Exception as e:
        print('missing', f)

## Downscaled product: a wet day

In [ ]:
import glob
ds = xr.open_dataset(sorted(glob.glob(f"{cfg['paths']['results']}/imerg_downscaled_1km_*.nc"))[-1])
aorc = xr.open_dataset(f"{cfg['paths']['processed']}/aorc_aligned_1km.nc", chunks={'time': 64})['precip']
imerg = xr.open_dataset(f"{cfg['paths']['processed']}/imerg_aligned_10km.nc")['precip']
val = ds.precipitation.sel(time=slice(cfg['time']['val_start'], cfg['time']['val_end']))
day = val.time[int(val.mean(('lat', 'lon')).argmax())].values
print('Wettest validation day:', str(day)[:10])
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
vmax = float(aorc.sel(time=day).quantile(0.995))
imerg.sel(time=day).plot(ax=axes[0], cmap=SEQ_CMAP, vmin=0, vmax=vmax); axes[0].set_title('IMERG 10 km')
val.sel(time=day).plot(ax=axes[1], cmap=SEQ_CMAP, vmin=0, vmax=vmax); axes[1].set_title('Downscaled 1 km')
aorc.sel(time=day).plot(ax=axes[2], cmap=SEQ_CMAP, vmin=0, vmax=vmax); axes[2].set_title('AORC 1 km')
plt.tight_layout()

## Uncertainty layer and model metadata

In [ ]:
if 'uncertainty' in ds:
    ds.uncertainty.plot(cmap=SEQ_CMAP, figsize=(6, 4)); plt.title('Std of 1 km residuals (training period, mm/day)')
meta = json.load(open(f"{cfg['paths']['models']}/model_metadata.json"))
print(json.dumps({k: meta[k] for k in ('model_type', 'hyperparams', 'best_iteration', 'n_train_rows', 'metrics', 'top10_features')}, indent=2))